# Part 2.6 Skills

After this notebook you can put a long procedure in a skill, so the agent loads it only
when it needs it.

In [ ]:
from kea.config import MODEL, setup

setup("kea-rebooking-local")

In [ ]:
from uuid import uuid4

from langchain_core.messages import HumanMessage
from langgraph.types import Command


async def ask(agent, text, thread_id=None):
    """Send one message. If the agent pauses for approval, approve and carry on."""
    config = {"configurable": {"thread_id": thread_id or str(uuid4())}}
    result = await agent.ainvoke({"messages": [HumanMessage(text)]}, config)
    while result.get("__interrupt__"):
        requests = result["__interrupt__"][0].value["action_requests"]
        for request in requests:
            print(f"Paused for approval: {request['name']}({request['args']})")
        decisions = [{"type": "approve"}] * len(requests)
        result = await agent.ainvoke(Command(resume={"decisions": decisions}), config)
    return result


def steps(result):
    """Print the tools the agent called, in order."""
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            args = ", ".join(f"{k}={str(v)[:40]!r}" for k, v in call["args"].items())
            print(f"  {call['name']}({args})")


def final(result):
    print(result["messages"][-1].text)

## See it

Ask the finished agent what a passenger is owed. The policy is a skill, not part of the
prompt. Look for the `read_file` of the skill. The agent may also hand the question to a
subagent with `task`, which [2.7](07_subagents.ipynb) covers.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from kea.rebooking.agent import build_agent

finished = build_agent(tools=[], checkpointer=InMemorySaver(), store=InMemoryStore())

QUESTION = (
    "KA 1182 was cancelled for a mechanical problem, and the next flight I can take "
    "leaves 145 minutes later. What credit am I owed?"
)
result = await ask(finished, QUESTION)
steps(result)
print()
final(result)

<img src="../../assets/screenshots/p2-06-skill-load.png" alt="the trace of the build-it agent below: the first model call with only the skill's name and description, then readfile on /skills/rebooking-policy/SKILL.md." width="900">

## Build it

Rebooking policy is long and only matters sometimes. Putting it in `AGENTS.md` would cost
tokens on every call. A **skill** is a folder with a `SKILL.md`. The agent sees only its
name and description until it decides it needs the rest, then reads the file. This is the
skill:

In [ ]:
from kea.rebooking.agent import HERE

print((HERE / "skills" / "rebooking-policy" / "SKILL.md").read_text())

In [ ]:
from deepagents import create_deep_agent

from kea.rebooking.agent import build_backend

skills_agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are Kea Air's rebooking assistant.",
    backend=build_backend(),
    skills=["/skills/"],
)

result = await ask(skills_agent, QUESTION)
steps(result)
print()
final(result)

## Under the hood

- `skills=["/skills/"]` is a source path on the backend. Every folder under it with a
  `SKILL.md` is a skill. Sources load in order, and a later one wins on a name clash.
- **Progressive disclosure is a file read.** Only `name` and `description` are in the
  prompt, so the `description` is the only thing the model has to decide with. Write it
  for a chooser: say when to use the skill, not what it is.
- **`AGENTS.md` is always loaded, a skill is loaded on demand.** Rules that apply to every
  conversation go in the first. Procedures that apply to some go in the second.
- A skill can carry more than text: scripts and reference files in the same folder, which
  the agent reads when the skill tells it to.

Docs: [Deep Agents skills](https://docs.langchain.com/oss/python/deepagents/skills).

## Recap

- A skill is a folder with a `SKILL.md`. Only its name and description are always in the
  prompt.
- The description is how the model chooses, so write it for a chooser.
- Always-on rules go in `AGENTS.md`. Occasional procedures go in skills.

**Next:** [Part 2.7 Subagents](07_subagents.ipynb)